# Uporaba modelov družine Meta  

## Uvod  

Ta lekcija bo zajemala: 

- Raziskovanje dveh glavnih modelov družine Meta - Llama 3.1 in Llama 3.2, plus trenutno razpoložljivih alternativ, ki se uporabljajo v primerih kode (Llama 3.3 in Llama 4) 
- Razumevanje primerov uporabe in scenarijev za vsak model 
- Primer kode, ki prikazuje unikatne značilnosti vsakega modela 


## Družina modelov Meta  

V tej lekciji bomo raziskali 2 modela iz družine Meta ali »Llama Herd« - Llama 3.1 in Llama 3.2. Zaženljive kode primere uporabljamo z Llama 3.3 in Llama 4, ki sta trenutno na voljo kot alternativi v katalogu Microsoft Foundry Models. 

Ti modeli so na voljo v različnih variantah v [Microsoft Foundry Models katalogu](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Opomba:** GitHub modeli se bodo upokojili konec julija 2026. Tukaj je več podrobnosti o uporabi [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) za prototipiranje z AI modeli.

> **Opomba:** Specifične različice modelov, na katerih je bila ta lekcija prvotno napisana (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) so bile kasneje upokojene iz kataloga Microsoft Foundry Models. Primeri kode spodaj uporabljajo `Llama-3.3-70B-Instruct` in `Llama-4-Scout-17B-16E-Instruct` kot trenutno razpoložljivi alternativi v katalogu Microsoft Foundry Models, zato še vedno delujejo danes.

Varijante modelov (kot je bilo prvotno pokrito v tej lekciji): 
- Llama 3.1 - 70B Instruct (upokojen) 
- Llama 3.1 - 405B Instruct (upokojen) 
- Llama 3.2 - 11B Vision Instruct (upokojen) 
- Llama 3.2 - 90B Vision Instruct (upokojen) 

Trenutno razpoložljive alternative, uporabljene v primerih kode: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Opomba: zgoraj omenjene različice Llama 3 se hranijo iz zgodovinskega konteksta in niso več na voljo v Microsoft Foundry Models.* 



## Llama 3.1 

Pri 405 milijardah parametrov spada Llama 3.1 v kategorijo odprtokodnih LLM. 

Model je nadgradnja prejšnje različice Llama 3 z zagotavljanjem: 

- Večje kontekstno okno - 128k tokenov proti 8k tokenom 
- Večja največja izhodna število tokenov - 4096 proti 2048 
- Boljša večjezična podpora - zaradi povečanja števila učnih tokenov 

To omogoča Llama 3.1 upravljanje bolj zapletenih primerov uporabe pri gradnji GenAI aplikacij, vključno z: 
- Naravni klic funkcij - možnost klica zunanjih orodij in funkcij zunaj LLM delovnega toka
- Boljša učinkovitost RAG - zaradi večjega kontekstnega okna 
- Generiranje sintetičnih podatkov - možnost ustvarjanja učinkovitih podatkov za naloge, kot je fino prilagajanje 



### Klicanje izvornih funkcij

Llama 3.1 je bila dodatno prilagojena, da je učinkovitejša pri klicih funkcij ali orodij. Vsebuje tudi dve vgrajeni orodji, ki jih model prepozna kot potrebni za uporabo glede na uporabnikov poziv. Ti orodji sta:

- **Brave Search** - Lahko se uporablja za pridobivanje najnovejših informacij, kot je vreme, z izvajanjem spletnega iskanja
- **Wolfram Alpha** - Lahko se uporablja za bolj zapletene matematične izračune, zato ni treba pisati lastnih funkcij.

Prav tako lahko ustvarite svoja lastna prilagojena orodja, ki jih lahko LLM kliče.

V spodnjem primeri kode:

- Definiramo razpoložljiva orodja (brave_search, wolfram_alpha) v sistemskem pozivu.
- Pošljemo uporabniški poziv, ki sprašuje o vremenu v določenem mestu.
- LLM bo odgovoril s klicem orodja Brave Search, ki bo izgledal tako `<|python_tag|>brave_search.call(query="Stockholm weather")`

*Opomba: Ta primer samo opravi klic orodja, če želite pridobiti rezultate, boste morali ustvariti brezplačen račun na Brave API strani in definirati funkcijo samo.


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Čeprav je LLM, je ena omejitev Llama 3.1 multimodalnost. To pomeni, da lahko uporablja različne vrste vhodov, kot so slike kot pozivi, in zagotavlja odgovore. Ta zmožnost je ena glavnih značilnosti Llama 3.2. Te značilnosti vključujejo tudi: 

- Multimodalnost - ima sposobnost vrednotenja tako besedilnih kot slikovnih pozivov 
- Majhne do srednje velike različice (11B in 90B) - to omogoča prilagodljive možnosti uvajanja, 
- Samo besedilne različice (1B in 3B) - to omogoča uvajanje modela na napravah na robu ali mobilnih napravah in zagotavlja nizko zakasnitev 

Podpora za multimodalnost predstavlja velik korak v svetu odprtokodnih modelov. Spodnji primer kode uporablja tako sliko kot besedilni poziv za analizo slike. Ker je bil Llama 3.2 90B Vision Instruct upokojena, primer uporablja `Llama-4-Scout-17B-16E-Instruct`, trenutno na voljo nativno multimodalno alternativo. 

### Podpora za multimodalnost z Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Učenje se tukaj ne konča, nadaljujte potovanje

Po zaključku te lekcije si oglejte našo [zbirko Učenja generativne umetne inteligence](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), da nadaljujete z nadgrajevanjem svojega znanja o generativni umetni inteligenci!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
